In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import os
import re
import unicodedata
from io import BytesIO
from datetime import datetime

In [ ]:
# ==============================================================================
# BLOCO DE RESTAURAÇÃO DO ETL 2
# ==============================================================================

import os
import sqlite3
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')

# Configurar caminhos
DRIVE_BASE_PATH = "/content/drive/MyDrive/Backup_ETL2_Recife"
db_path = os.path.join(DRIVE_BASE_PATH, "infracoes_recife.db")

# Verificar se o banco já existe no Drive
if os.path.exists(db_path):
    print("="*60)
    print("✅ BANCO DE DADOS ENCONTRADO NO DRIVE!")
    print("="*60)
    print(f"📁 Local: {db_path}")
    print("\n🔄 Carregando dados diretamente do banco SQLite...")

    # Conectar ao banco
    con = sqlite3.connect(db_path)

    # Carregar fato_infracoes
    fato_infracoes = pd.read_sql("SELECT * FROM fato_infracoes", con)

    # Carregar dimensões (se necessário para análises)
    dim_tempo = pd.read_sql("SELECT * FROM dim_tempo", con)
    dim_infracao = pd.read_sql("SELECT * FROM dim_infracao", con)
    dim_local = pd.read_sql("SELECT * FROM dim_local", con)

    con.close()

    print(f"✅ Dados restaurados com sucesso!")
    print(f"   - Fato infrações: {len(fato_infracoes)} registros")
    print(f"   - Dimensão tempo: {len(dim_tempo)} registros")
    print(f"   - Dimensão infração: {len(dim_infracao)} registros")
    print(f"   - Dimensão local: {len(dim_local)} registros")
    print("\n Você pode pular os blocos de extração e processamento.")
    print("   Prossega diretamente para as análises e indicadores.")
    print("="*60)

    # Flag para indicar que os dados foram restaurados
    DADOS_RESTAURADOS = True

else:
    print("="*60)
    print("⚠️ BANCO DE DADOS NÃO ENCONTRADO")
    print("="*60)
    print(f"📁 Local esperado: {db_path}")
    print("\n❗ Será necessário executar o pipeline completo:")
    print("   1. Extração dos dados da API")
    print("   2. Transformação e limpeza")
    print("   3. Carga no banco SQLite")
    print("\n💡 Após a primeira execução, os dados serão salvos no Drive")
    print("   e você poderá restaurá-los nas próximas execuções.")
    print("="*60)

    # Flag para indicar que precisa executar o pipeline
    DADOS_RESTAURADOS = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BANCO DE DADOS ENCONTRADO NO DRIVE!
📁 Local: /content/drive/MyDrive/Backup_ETL2_Recife/infracoes_recife.db

🔄 Carregando dados diretamente do banco SQLite...
✅ Dados restaurados com sucesso!
   - Fato infrações: 404975 registros
   - Dimensão tempo: 1348 registros
   - Dimensão infração: 255 registros
   - Dimensão local: 56 registros

 Você pode pular os blocos de extração e processamento.
   Prossega diretamente para as análises e indicadores.


In [ ]:
from google.colab import drive
import os

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Definir pasta de backup no Drive
DRIVE_BASE_PATH = "/content/drive/MyDrive/Backup_ETL2_Recife"
os.makedirs(DRIVE_BASE_PATH, exist_ok=True)

# 3. Criar subpastas organizadas
DIR_RAW = os.path.join(DRIVE_BASE_PATH, "raw")
DIR_PROCESSED = os.path.join(DRIVE_BASE_PATH, "processed")
DIR_REPORTS = os.path.join(DRIVE_BASE_PATH, "reports")

for directory in [DIR_RAW, DIR_PROCESSED, DIR_REPORTS]:
    os.makedirs(directory, exist_ok=True)

print("="*60)
print("✅ BLOCO 2 CARREGADO COM SUCESSO!")
print(f"📁 Os dados brutos, tratados e relatórios serão salvos em:")
print(f"   {DRIVE_BASE_PATH}")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BLOCO 2 CARREGADO COM SUCESSO!
📁 Os dados brutos, tratados e relatórios serão salvos em:
   /content/drive/MyDrive/Backup_ETL2_Recife


In [ ]:
resource_id = "fd97260b-4a91-4c6a-ad14-7285e5f6ed9a"

url_api = "https://dados.recife.pe.gov.br/api/3/action/resource_show"

resposta = requests.get(
    url_api,
    params={"id": resource_id},
    timeout=60
)

resposta.raise_for_status()

dados_api = resposta.json()

if not dados_api["success"]:
    raise Exception("Não foi possível consultar a fonte oficial.")

recurso = dados_api["result"]

print("Nome:", recurso["name"])
print("Formato:", recurso["format"])
print("URL:", recurso["url"])
print("Última atualização:", recurso.get("last_modified"))

Nome: Registro das infrações de Trânsito - 2024
Formato: CSV
URL: https://dados.recife.pe.gov.br:443/dataset/fbe0c549-cd14-4146-886c-38bc0859cd2f/resource/fd97260b-4a91-4c6a-ad14-7285e5f6ed9a/download/registro-das-infracoes-de-transito-2024.csv
Última atualização: 2026-03-06T17:32:18.959093


In [ ]:
url_csv = recurso["url"]

resposta_csv = requests.get(url_csv, timeout=180)

resposta_csv.raise_for_status()

df_original = pd.read_csv(
    BytesIO(resposta_csv.content),
    sep=None,
    engine="python",
    encoding="utf-8",
    dtype=str
)

print("Dados extraídos com sucesso!")
print("Total de registros:", len(df_original))
print("Total de colunas:", len(df_original.columns))

display(df_original.head())

Dados extraídos com sucesso!
Total de registros: 404975
Total de colunas: 8


,datainfracao,horainfracao,dataimplantacao,agenteequipamento,infracao,descricaoinfracao,amparolegal,localcometimento
0,01/01/2024 00:00,00:59:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5479,Estacionar o veículo impedindo a movimentação ...,"Art. 181, Inc. X","AVENIDA BOA VIAGEM, LADO OPOSTO AO N. 618"
1,01/01/2024 00:00,01:08:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,6564,Conduzir o veículo transportando passageiros e...,"Art. 230, Inc. II","AVENIDA BOA VIAGEM, EM FRENTE AO N. 500. SENTI..."
2,01/01/2024 00:00,01:50:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5452,Estacionar o veículo no passeio.,"Art. 181, Inc. VIII","RUA CAPITAO REBELINHO, EM FRENTE AO N. 482, SE..."
3,01/01/2024 00:00,02:00:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5452,Estacionar o veículo no passeio.,"Art. 181, Inc. VIII","RUA CAPITAO REBELINHO, EM FRENTE AO N. 482"
4,01/01/2024 00:00,02:18:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,6050,"Avançar o sinal vermelho do semáforo, exceto o...",Art. 208,"AVENIDA CONSELHEIRO AGUIAR, SOB AO SEMAFORO N...."


In [ ]:
print("COLUNAS DISPONÍVEIS:")

for coluna in df_original.columns:
    print(coluna)

COLUNAS DISPONÍVEIS:
datainfracao
horainfracao
dataimplantacao
agenteequipamento
infracao
descricaoinfracao
amparolegal
localcometimento


In [ ]:
df = df_original.copy()

print("Cópia criada.")
print("Registros:", len(df))

Cópia criada.
Registros: 404975


In [ ]:
def padronizar_nome(nome):
    nome = str(nome).strip().lower()

    nome = unicodedata.normalize("NFKD", nome)
    nome = "".join(
        c for c in nome
        if not unicodedata.combining(c)
    )

    nome = re.sub(r"[^a-z0-9]+", "_", nome)

    return nome.strip("_")

df.columns = [
    padronizar_nome(coluna)
    for coluna in df.columns
]

print(df.columns.tolist())

['datainfracao', 'horainfracao', 'dataimplantacao', 'agenteequipamento', 'infracao', 'descricaoinfracao', 'amparolegal', 'localcometimento']


In [ ]:
df = df.replace(r"^\s*$", np.nan, regex=True)

auditoria_nulos = pd.DataFrame({
    "coluna": df.columns,
    "quantidade_nulos": [
        df[col].isna().sum()
        for col in df.columns
    ],
    "percentual_nulos": [
        round(df[col].isna().mean() * 100, 2)
        for col in df.columns
    ]
})

display(auditoria_nulos)

,coluna,quantidade_nulos,percentual_nulos
0,datainfracao,0,0.0
1,horainfracao,0,0.0
2,dataimplantacao,0,0.0
3,agenteequipamento,0,0.0
4,infracao,0,0.0
5,descricaoinfracao,0,0.0
6,amparolegal,0,0.0
7,localcometimento,0,0.0


In [ ]:
df["datainfracao"] = pd.to_datetime(
    df["datainfracao"],
    errors="coerce",
    dayfirst=True
)

print("Datas convertidas.")

print(
    "Datas inválidas:",
    df["datainfracao"].isna().sum()
)

Datas convertidas.
Datas inválidas: 0


In [ ]:
total_original = len(df)

df_2024 = df[
    df["datainfracao"].dt.year == 2024
].copy()

print("Total original:", total_original)
print("Registros de 2024:", len(df_2024))
print("Registros fora do recorte ou sem data válida:",
      total_original - len(df_2024))

Total original: 404975
Registros de 2024: 404975
Registros fora do recorte ou sem data válida: 0


In [ ]:
df_2024["horainfracao"] = (
    df_2024["horainfracao"]
    .astype("string")
    .str.strip()
)

df_2024["hora_convertida"] = pd.to_datetime(
    df_2024["horainfracao"],
    format="%H:%M:%S",
    errors="coerce"
)

# Tentar também o formato HH:MM
faltantes = df_2024["hora_convertida"].isna()

df_2024.loc[faltantes, "hora_convertida"] = pd.to_datetime(
    df_2024.loc[faltantes, "horainfracao"],
    format="%H:%M",
    errors="coerce"
)

df_2024["hora"] = df_2024["hora_convertida"].dt.hour

print("Horários tratados.")
print("Horários inválidos:",
      df_2024["hora"].isna().sum())

Horários tratados.
Horários inválidos: 0


In [ ]:
colunas_texto = [
    "agenteequipamento",
    "descricaoinfracao",
    "amparolegal",
    "localcometimento"
]

for coluna in colunas_texto:
    df_2024[coluna] = (
        df_2024[coluna]
        .astype("string")
        .str.strip()
    )

print("Campos textuais padronizados.")

Campos textuais padronizados.


In [ ]:
df_2024["infracao"] = pd.to_numeric(
    df_2024["infracao"],
    errors="coerce"
)

print("Códigos de infração convertidos.")
print("Códigos inválidos:",
      df_2024["infracao"].isna().sum())

Códigos de infração convertidos.
Códigos inválidos: 0


In [ ]:
# ==============================================================================
# CORREÇÃO #4: Padronização temporal em Português (compatível com ETL 1)
# ==============================================================================

df_2024["ano"] = df_2024["datainfracao"].dt.year
df_2024["mes_num"] = df_2024["datainfracao"].dt.month

# Mapeamento de meses em Português (com numeração, igual ao ETL 1)
meses_pt = {
    1: "01 - Janeiro", 2: "02 - Fevereiro", 3: "03 - Março", 4: "04 - Abril",
    5: "05 - Maio", 6: "06 - Junho", 7: "07 - Julho", 8: "08 - Agosto",
    9: "09 - Setembro", 10: "10 - Outubro", 11: "11 - Novembro", 12: "12 - Dezembro"
}
df_2024["nome_mes"] = df_2024["mes_num"].map(meses_pt)

# Mapeamento de dias da semana em Português (0 = Segunda, 6 = Domingo)
dias_semana_pt = {
    0: "Segunda-feira", 1: "Terça-feira", 2: "Quarta-feira", 3: "Quinta-feira",
    4: "Sexta-feira", 5: "Sábado", 6: "Domingo"
}
df_2024["dia_semana_num"] = df_2024["datainfracao"].dt.dayofweek
df_2024["dia_semana"] = df_2024["dia_semana_num"].map(dias_semana_pt)

# Faixas horárias padronizadas (igual ao ETL 1)
bins = [-1, 5, 11, 17, 23]
labels = ["00h-05h (Madrugada)", "06h-11h (Manhã)", "12h-17h (Tarde)", "18h-23h (Noite)"]
df_2024["faixa_horaria"] = pd.cut(df_2024["hora"], bins=bins, labels=labels)

# Remover coluna 'mes' antiga (numérica) se existir, para evitar duplicidade de informação
if "mes" in df_2024.columns and "mes_num" in df_2024.columns:
    df_2024 = df_2024.drop(columns=["mes"])

print("✅ Padronização temporal concluída!")
print("\n📊 Amostra das colunas padronizadas:")
display(df_2024[["datainfracao", "hora", "ano", "mes_num", "nome_mes", "dia_semana", "faixa_horaria"]].head(10))

✅ Padronização temporal concluída!

📊 Amostra das colunas padronizadas:


,datainfracao,hora,ano,mes_num,nome_mes,dia_semana,faixa_horaria
0,2024-01-01,0,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
1,2024-01-01,1,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
2,2024-01-01,1,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
3,2024-01-01,2,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
4,2024-01-01,2,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
5,2024-01-01,2,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
6,2024-01-01,2,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
7,2024-01-01,3,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
8,2024-01-01,3,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)
9,2024-01-01,3,2024,1,01 - Janeiro,Segunda-feira,00h-05h (Madrugada)


In [ ]:
# ==============================================================================
# CORREÇÃO #5: Padronização dos nomes das colunas (melhor legibilidade)
# ==============================================================================

mapeamento_colunas = {
    "datainfracao": "data_infracao",
    "horainfracao": "hora_infracao",
    "dataimplantacao": "data_implantacao",
    "agenteequipamento": "agente_equipamento",
    "descricaoinfracao": "descricao_infracao",
    "amparolegal": "amparo_legal",
    "localcometimento": "local_cometimento",
    "hora_convertida": "hora_completa"
}

# Aplica a renomeação (apenas se a coluna existir)
colunas_existentes = {k: v for k, v in mapeamento_colunas.items() if k in df_2024.columns}
df_2024 = df_2024.rename(columns=colunas_existentes)

print("✅ Padronização dos nomes das colunas concluída!")
print("\n📋 Colunas renomeadas:")
for antigo, novo in colunas_existentes.items():
    print(f"   {antigo} → {novo}")

print("\n📋 Colunas finais do dataset:")
print(df_2024.columns.tolist())

✅ Padronização dos nomes das colunas concluída!

📋 Colunas renomeadas:
   datainfracao → data_infracao
   horainfracao → hora_infracao
   dataimplantacao → data_implantacao
   agenteequipamento → agente_equipamento
   descricaoinfracao → descricao_infracao
   amparolegal → amparo_legal
   localcometimento → local_cometimento
   hora_convertida → hora_completa

📋 Colunas finais do dataset:
['data_infracao', 'hora_infracao', 'data_implantacao', 'agente_equipamento', 'infracao', 'descricao_infracao', 'amparo_legal', 'local_cometimento', 'hora_completa', 'hora', 'ano', 'mes_num', 'nome_mes', 'dia_semana_num', 'dia_semana', 'faixa_horaria']


In [ ]:
import unicodedata
import re
import numpy as np

print("🔄 Iniciando extração de bairros em 2 etapas...")

# ==============================================================================
# ETAPA 1: Extração Robusta por Nome do Bairro
# ==============================================================================
bairros_recife = [
    "Afogados", "Aflitos", "Água Fria", "Alto José do Pinho", "Alto José Bonifácio",
    "Apipucos", "Arruda", "Barro", "Beberibe", "Boa Viagem", "Boa Vista", "Bombas",
    "Brejo da Guabiraba", "Brasília Teimosa", "Cabanga", "Cajueiro", "Campina do Barreto",
    "Campo Grande", "Casa Amarela", "Casa Forte", "Caxangá", "Coelhos", "Cordeiro",
    "Derby", "Dois Irmãos", "Dois Unidos", "Encruzilhada", "Engenho do Meio", "Espinheiro",
    "Fundão", "Graças", "Guabiraba", "Ibura", "Ilha do Leite", "Ilha do Retiro", "Imbiribeira",
    "Iputinga", "Ipsep", "Jaqueira", "Jardim São Paulo", "Jordão", "Macaxeira", "Madalena",
    "Mangabeira", "Monteiro", "Mustardinha", "Parnamirim", "Pina", "Poço da Panela", "Prado",
    "Rosarinho", "San Martin", "Santo Amaro", "Santo Antônio", "São José",
    "Tamarineira", "Torre", "Torreão", "Várzea", "Vasco da Gama", "Recife"
]

def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto.lower()

bairros_normalizados = {normalizar_texto(b): b for b in bairros_recife}
bairros_ordenados = sorted(bairros_normalizados.keys(), key=len, reverse=True)
pattern_bairros = r'\b(' + '|'.join(bairros_ordenados) + r')\b'

def extrair_bairro_robusto(local):
    if pd.isna(local):
        return "NÃO INFORMADO"
    local_norm = normalizar_texto(local)
    match = re.search(pattern_bairros, local_norm)
    if match:
        return bairros_normalizados[match.group(0)].title()
    return "OUTRO / NÃO IDENTIFICADO"

# ⚠️ ATENÇÃO: Use 'local_cometimento' (com underscore) em vez de 'localcometimento'
df_2024['bairro'] = df_2024['local_cometimento'].apply(extrair_bairro_robusto)

# ==============================================================================
# ETAPA 2: Fallback por Palavras-chave de Ruas/Referências Famosas
# ==============================================================================
print("📍 Aplicando mapeamento de fallback para ruas e referências...")

mask_nao_identificado = (df_2024['bairro'] == "OUTRO / NÃO IDENTIFICADO")
# ⚠️ ATENÇÃO: Use 'local_cometimento' aqui também
local_lower = df_2024.loc[mask_nao_identificado, 'local_cometimento'].astype(str).str.lower()

fallback_map = {
    "capitao rebelinho": "Imbiribeira",
    "cais santa rita": "São José",
    "real da torre": "Torre",
    "arnaldo carneiro": "Casa Amarela",
    "dom joao vi": "Boa Viagem",
    "engenheiro domingos ferreira": "Boa Viagem",
    "conselheiro aguiar": "Boa Viagem",
    "barao de souza leao": "Boa Viagem",
    "projecao a": "Boa Viagem",
    "projecao b": "Boa Viagem",
    "ribeiro de brito": "Santo Amaro",
    "do imperador": "Santo Antônio",
    "da aurora": "Santo Antônio",
    "do hospicio": "Santo Antônio",
    "da concordia": "São José",
    "amaro bezerra": "Cordeiro",
    "professor morais": "Santo Amaro",
    "joao de barros": "Santo Amaro",
    "severo dourado": "Santo Amaro",
    "agamenon magalhaes": "Boa Viagem",
    "caxanga": "Caxangá"
}

for keyword, bairro_correto in fallback_map.items():
    condicao = mask_nao_identificado & (local_lower.str.contains(keyword, na=False))
    df_2024.loc[condicao, 'bairro'] = bairro_correto

# Padronização final
df_2024['bairro'] = df_2024['bairro'].replace("OUTRO / NÃO IDENTIFICADO", "Não Identificado")
df_2024['bairro'] = df_2024['bairro'].replace("NÃO INFORMADO", "Não Identificado")

# ==============================================================================
# RESULTADO FINAL
# ==============================================================================
print("\n✅ Extração concluída!")
print("\n📊 Top 15 Bairros mais frequentes encontrados:")
print(df_2024['bairro'].value_counts().head(15))

nao_id = (df_2024['bairro'] == "Não Identificado").sum()
print(f"\n⚠️ Registros que ainda não foram identificados: {nao_id} ({(nao_id/len(df_2024)*100):.2f}%)")
print("💡 Dica: Este percentual restante é uma limitação real dos dados de origem (endereço apenas por rua) e deve ser citado no relatório final.")

🔄 Iniciando extração de bairros em 2 etapas...
📍 Aplicando mapeamento de fallback para ruas e referências...

✅ Extração concluída!

📊 Top 15 Bairros mais frequentes encontrados:
bairro
Não Identificado    220334
Boa Viagem           58497
Caxangá              39092
Boa Vista            27232
Recife               14637
Torre                 9733
São José              5918
Derby                 5072
Casa Amarela          4994
Afogados              4897
Beberibe              4498
Pina                  2721
Santo Antônio         1728
Água Fria             1108
Santo Amaro            787
Name: count, dtype: int64

⚠️ Registros que ainda não foram identificados: 220334 (54.41%)
💡 Dica: Este percentual restante é uma limitação real dos dados de origem (endereço apenas por rua) e deve ser citado no relatório final.


In [ ]:
total_registros = len(df_2024)

sem_local = df_2024["local_cometimento"].isna().sum()

com_local = total_registros - sem_local

percentual_sem_local = (
    sem_local / total_registros * 100
    if total_registros > 0 else 0
)

print("Total de registros:", total_registros)
print("Com local informado:", com_local)
print("Sem local informado:", sem_local)

print(
    "Percentual sem local:",
    round(percentual_sem_local, 2), "%"
)

Total de registros: 404975
Com local informado: 404975
Sem local informado: 0
Percentual sem local: 0.0 %


In [ ]:
colunas_comparacao = [
    "data_infracao",
    "hora_infracao",
    "agente_equipamento",
    "infracao",
    "descricao_infracao",
    "local_cometimento"
]

repetidos = df_2024.duplicated(
    subset=colunas_comparacao,
    keep=False
)

print("Linhas envolvidas em possíveis repetições:",
      repetidos.sum())

display(df_2024[repetidos].head(10))

Linhas envolvidas em possíveis repetições: 15241


,data_infracao,hora_infracao,data_implantacao,agente_equipamento,infracao,descricao_infracao,amparo_legal,local_cometimento,hora_completa,hora,ano,mes_num,nome_mes,dia_semana_num,dia_semana,faixa_horaria,bairro
5,2024-01-01,02:40:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5452,Estacionar o veículo no passeio.,"Art. 181, Inc. VIII","AVENIDA ENGENHEIRO DOMINGOS FERREIRA, EM FRENT...",1900-01-01 02:40:00,2,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Boa Viagem
6,2024-01-01,02:40:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5452,Estacionar o veículo no passeio.,"Art. 181, Inc. VIII","AVENIDA ENGENHEIRO DOMINGOS FERREIRA, EM FRENT...",1900-01-01 02:40:00,2,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Boa Viagem
371,2024-01-01,14:59:00,03/01/2024 00:00,Código 7 - ZONA AZUL TALÃO ELETRÔNICO,5541,Estacionar o veículo em desacordo com as condi...,"Art. 181, Inc. XVII","AVENIDA BOA VIAGEM, EM FRENTE AO N. 4334",1900-01-01 14:59:00,14,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Boa Viagem
372,2024-01-01,14:59:00,03/01/2024 00:00,Código 7 - ZONA AZUL TALÃO ELETRÔNICO,5541,Estacionar o veículo em desacordo com as condi...,"Art. 181, Inc. XVII","AVENIDA BOA VIAGEM, EM FRENTE AO N. 4334",1900-01-01 14:59:00,14,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Boa Viagem
373,2024-01-01,14:59:00,03/01/2024 00:00,Código 7 - ZONA AZUL TALÃO ELETRÔNICO,5541,Estacionar o veículo em desacordo com as condi...,"Art. 181, Inc. XVII","AVENIDA BOA VIAGEM, EM FRENTE AO N. 4334",1900-01-01 14:59:00,14,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Boa Viagem
374,2024-01-01,14:59:00,03/01/2024 00:00,Código 7 - ZONA AZUL TALÃO ELETRÔNICO,5541,Estacionar o veículo em desacordo com as condi...,"Art. 181, Inc. XVII","AVENIDA BOA VIAGEM, EM FRENTE AO N. 4334",1900-01-01 14:59:00,14,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Boa Viagem
401,2024-01-01,15:27:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5550,Estacionar o veículo em locais e horários proi...,"Art. 181, Inc. XVIII","AVENIDA PREFEITO ANTONIO PEREIRA, AO LADO PORT...",1900-01-01 15:27:00,15,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Não Identificado
402,2024-01-01,15:27:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5550,Estacionar o veículo em locais e horários proi...,"Art. 181, Inc. XVIII","AVENIDA PREFEITO ANTONIO PEREIRA, AO LADO PORT...",1900-01-01 15:27:00,15,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Não Identificado
403,2024-01-01,15:27:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5550,Estacionar o veículo em locais e horários proi...,"Art. 181, Inc. XVIII","AVENIDA PREFEITO ANTONIO PEREIRA, AO LADO PORT...",1900-01-01 15:27:00,15,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Não Identificado
404,2024-01-01,15:27:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5550,Estacionar o veículo em locais e horários proi...,"Art. 181, Inc. XVIII","AVENIDA PREFEITO ANTONIO PEREIRA, AO LADO PORT...",1900-01-01 15:27:00,15,2024,1,01 - Janeiro,0,Segunda-feira,12h-17h (Tarde),Não Identificado


In [ ]:
indicador_tipo = (
    df_2024
    .groupby(
        ["infracao", "descricao_infracao"],  # ⚠️ Nome atualizado
        dropna=False
    )
    .size()
    .reset_index(name="quantidade")
    .sort_values("quantidade", ascending=False)
)

indicador_tipo["percentual"] = (
    indicador_tipo["quantidade"]
    / len(df_2024) * 100
).round(2)

display(indicador_tipo.head(20))

,infracao,descricao_infracao,quantidade,percentual
236,7455,Transitar em velocidade superior à máxima perm...,95078,23.48
240,7587,Transitar na faixa ou via exclusiva regulam. ...,69676,17.21
60,5541,Estacionar o veículo em desacordo com as condi...,54673,13.50
90,5746,Transitar em locais e horários não permitidos ...,23973,5.92
64,5550,Estacionar o veículo em locais e horários proi...,20769,5.13
138,6050,Avançar o sinal vermelho do semáforo - fiscali...,14970,3.70
229,7340,Dirigir o veículo usando calçado que não se fi...,12634,3.12
244,7633,Dirigir o veículo segurando o telefone celular.,10697,2.64
82,5673,Parar o veículo sobre a faixa de pedestre na m...,10603,2.62
136,6041,Executar operação de conversão à esquerda em l...,9450,2.33


In [ ]:
indicador_horario = (
    df_2024
    .groupby("hora", dropna=False)
    .size()
    .reset_index(name="quantidade")
    .sort_values("hora")
)

display(indicador_horario)

,hora,quantidade
0,0,432
1,1,421
2,2,384
3,3,113
4,4,177
5,5,260
6,6,15485
7,7,23628
8,8,35235
9,9,41825


In [ ]:
indicador_mes = (
    df_2024
    .groupby("mes_num", dropna=False)
    .size()
    .reset_index(name="quantidade")
    .sort_values("mes_num")
)

display(indicador_mes)

,mes_num,quantidade
0,1,30376
1,2,24090
2,3,27892
3,4,32236
4,5,32145
5,6,31026
6,7,32842
7,8,33522
8,9,37101
9,10,36819


In [ ]:
indicador_dia_hora = pd.crosstab(
    df_2024["dia_semana"],
    df_2024["hora"]
)

display(indicador_dia_hora)

hora,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
dia_semana,,,,,,,,,,,,,,,,,,,,,
Domingo,104,134,111,32,46,67,2701,3449,3638,4100,...,4040,4315,3626,2376,1552,1659,1518,1566,210,192
Quarta-feira,28,29,38,16,22,23,2256,3847,5905,6623,...,4633,6522,4757,2617,2106,1638,1440,1734,373,117
Quinta-feira,14,17,44,8,12,28,2083,3311,5535,6612,...,4672,6423,4685,2673,2257,1594,1618,1860,546,204
Segunda-feira,73,72,61,17,17,25,2091,3364,5349,6201,...,4638,6442,4629,2447,1917,1375,1355,1640,400,102
Sexta-feira,75,50,49,7,26,49,2218,3613,5645,6652,...,5065,6448,5490,3485,2645,2284,1926,1992,482,245
Sábado,103,72,46,23,36,44,1974,2513,3550,5195,...,3245,3311,3020,1921,1219,1325,1590,1736,341,205
Terça-feira,35,47,35,10,18,24,2162,3531,5613,6442,...,4947,6885,4869,2678,2065,1413,1453,1763,350,124


In [ ]:
fato_infracoes = df_2024.copy()

fato_infracoes["origem_dados"] = (
    "Registro das Infrações de Trânsito - 2024"
)

fato_infracoes["resource_id"] = resource_id

print("Tabela fato_infracoes criada!")
print("Registros:", len(fato_infracoes))
print("Colunas:", len(fato_infracoes.columns))

display(fato_infracoes.head())

Tabela fato_infracoes criada!
Registros: 404975
Colunas: 19


,data_infracao,hora_infracao,data_implantacao,agente_equipamento,infracao,descricao_infracao,amparo_legal,local_cometimento,hora_completa,hora,ano,mes_num,nome_mes,dia_semana_num,dia_semana,faixa_horaria,bairro,origem_dados,resource_id
0,2024-01-01,00:59:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5479,Estacionar o veículo impedindo a movimentação ...,"Art. 181, Inc. X","AVENIDA BOA VIAGEM, LADO OPOSTO AO N. 618",1900-01-01 00:59:00,0,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Boa Viagem,Registro das Infrações de Trânsito - 2024,fd97260b-4a91-4c6a-ad14-7285e5f6ed9a
1,2024-01-01,01:08:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,6564,Conduzir o veículo transportando passageiros e...,"Art. 230, Inc. II","AVENIDA BOA VIAGEM, EM FRENTE AO N. 500. SENTI...",1900-01-01 01:08:00,1,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Boa Viagem,Registro das Infrações de Trânsito - 2024,fd97260b-4a91-4c6a-ad14-7285e5f6ed9a
2,2024-01-01,01:50:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5452,Estacionar o veículo no passeio.,"Art. 181, Inc. VIII","RUA CAPITAO REBELINHO, EM FRENTE AO N. 482, SE...",1900-01-01 01:50:00,1,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Imbiribeira,Registro das Infrações de Trânsito - 2024,fd97260b-4a91-4c6a-ad14-7285e5f6ed9a
3,2024-01-01,02:00:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,5452,Estacionar o veículo no passeio.,"Art. 181, Inc. VIII","RUA CAPITAO REBELINHO, EM FRENTE AO N. 482",1900-01-01 02:00:00,2,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Imbiribeira,Registro das Infrações de Trânsito - 2024,fd97260b-4a91-4c6a-ad14-7285e5f6ed9a
4,2024-01-01,02:18:00,03/01/2024 00:00,Código 8 - AUTOS NO TALÃO ELETRÔNICO,6050,"Avançar o sinal vermelho do semáforo, exceto o...",Art. 208,"AVENIDA CONSELHEIRO AGUIAR, SOB AO SEMAFORO N....",1900-01-01 02:18:00,2,2024,1,01 - Janeiro,0,Segunda-feira,00h-05h (Madrugada),Boa Viagem,Registro das Infrações de Trânsito - 2024,fd97260b-4a91-4c6a-ad14-7285e5f6ed9a


In [ ]:
import sqlite3

print("="*60)
print("🔧 CORREÇÃO #6: Gerando banco SQLite com modelo dimensional")
print("="*60)

# Caminho do banco no Drive
db_path = f"{DRIVE_BASE_PATH}/infracoes_recife.db"

# Conectar ao banco (cria se não existir)
con = sqlite3.connect(db_path)

# ==============================================================================
# DIMENSÃO TEMPO
# ==============================================================================
print("\n📅 Criando dimensão tempo...")
dim_tempo = (
    df_2024[["data_infracao", "mes_num", "nome_mes", "dia_semana_num", "dia_semana", "faixa_horaria"]]
    .drop_duplicates()
    .sort_values("data_infracao")
    .reset_index(drop=True)
)
dim_tempo.insert(0, "id_tempo", range(1, len(dim_tempo) + 1))
dim_tempo.to_sql("dim_tempo", con, if_exists="replace", index=False)
print(f"   ✅ {len(dim_tempo)} registros únicos de tempo")

# ==============================================================================
# DIMENSÃO INFRAÇÃO
# ==============================================================================
print("\n🚗 Criando dimensão infração...")
dim_infracao = (
    df_2024[["infracao", "descricao_infracao", "amparo_legal"]]
    .drop_duplicates()
    .sort_values("infracao")
    .reset_index(drop=True)
)
dim_infracao.insert(0, "id_infracao", range(1, len(dim_infracao) + 1))
dim_infracao.to_sql("dim_infracao", con, if_exists="replace", index=False)
print(f"   ✅ {len(dim_infracao)} tipos únicos de infração")

# ==============================================================================
# DIMENSÃO LOCAL (bairro) - CORRIGIDA
# ==============================================================================
print("\n📍 Criando dimensão local...")

# Pega os bairros únicos, excluindo nulos
bairros_unicos = sorted(df_2024["bairro"].dropna().unique().tolist())

# Garante que "Não Identificado" apareça apenas uma vez e com id_local = -1
if "Não Identificado" in bairros_unicos:
    bairros_unicos.remove("Não Identificado")

# Cria o DataFrame com "Não Identificado" primeiro
lista_bairros = ["Não Identificado"] + bairros_unicos
dim_local = pd.DataFrame({"bairro": lista_bairros})
dim_local.insert(0, "id_local", [-1] + list(range(1, len(dim_local))))

dim_local.to_sql("dim_local", con, if_exists="replace", index=False)
print(f"   ✅ {len(dim_local)} bairros únicos")

# ==============================================================================
# DIMENSÃO FONTE
# ==============================================================================
print("\n📋 Criando dimensão fonte...")
dim_fonte = pd.DataFrame([{
    "id_fonte": 1,
    "fonte": "Portal de Dados Abertos do Recife",
    "conjunto": "Registro das Infrações de Trânsito",
    "ano": 2024,
    "resource_id": resource_id,
    "data_extracao": datetime.now().isoformat(),
    "formato": "CSV"
}])
dim_fonte.to_sql("dim_fonte", con, if_exists="replace", index=False)
print(f"   ✅ 1 registro de fonte")

# ==============================================================================
# FATO INFRAÇÕES
# ==============================================================================
print("\n📊 Criando fato_infracoes...")

fato = df_2024.copy()

# Merge com dim_tempo
fato = fato.merge(dim_tempo, on=["data_infracao", "mes_num", "nome_mes", "dia_semana_num", "dia_semana", "faixa_horaria"], how="left")

# Merge com dim_infracao
fato = fato.merge(dim_infracao, on=["infracao", "descricao_infracao", "amparo_legal"], how="left")

# Merge com dim_local (bairro)
fato = fato.merge(dim_local, on="bairro", how="left")
fato["id_local"] = fato["id_local"].fillna(-1).astype(int)

# Selecionar colunas da fato
cols_fato = [
    "id_tempo", "id_infracao", "id_local",
    "data_infracao", "hora_infracao", "hora",
    "faixa_horaria", "mes_num", "nome_mes", "dia_semana",
    "infracao", "descricao_infracao", "amparo_legal",
    "local_cometimento", "bairro",
    "agente_equipamento",
    "resource_id", "origem_dados"
]

# Filtrar apenas colunas que existem
cols_existentes = [c for c in cols_fato if c in fato.columns]
fato_infracoes = fato[cols_existentes].copy()

fato_infracoes.to_sql("fato_infracoes", con, if_exists="replace", index=False)
print(f"   ✅ {len(fato_infracoes)} registros na fato_infracoes")

# Fechar conexão
con.close()

print("\n" + "="*60)
print("✅ BANCO SQLITE CRIADO COM SUCESSO!")
print(f"📁 Local: {db_path}")
print("="*60)

🔧 CORREÇÃO #6: Gerando banco SQLite com modelo dimensional

📅 Criando dimensão tempo...
   ✅ 1348 registros únicos de tempo

🚗 Criando dimensão infração...
   ✅ 255 tipos únicos de infração

📍 Criando dimensão local...
   ✅ 56 bairros únicos

📋 Criando dimensão fonte...
   ✅ 1 registro de fonte

📊 Criando fato_infracoes...
   ✅ 404975 registros na fato_infracoes

✅ BANCO SQLITE CRIADO COM SUCESSO!
📁 Local: /content/drive/MyDrive/Backup_ETL2_Recife/infracoes_recife.db


In [ ]:

relatorio_qualidade = pd.DataFrame({
    "indicador": [
        "Linhas extraídas",
        "Linhas carregadas",
        "Linhas fora do recorte ou com data inválida",
        "Datas inválidas no arquivo original",
        "Horários inválidos em 2024",
        "Códigos de infração inválidos em 2024",
        "Registros sem local",
        "Duplicidades exatas em todas as colunas",
        "Linhas envolvidas em possíveis repetições"
    ],

    "quantidade": [
        total_original,
        len(fato_infracoes),
        total_original - len(fato_infracoes),
        df["datainfracao"].isna().sum(),
        df_2024["hora"].isna().sum(),
        df_2024["infracao"].isna().sum(),
        sem_local,
        df_2024.duplicated().sum(),
        repetidos.sum()
    ]
})

display(relatorio_qualidade)

print("Relatório de qualidade corrigido!")


,indicador,quantidade
0,Linhas extraídas,404975
1,Linhas carregadas,404975
2,Linhas fora do recorte ou com data inválida,0
3,Datas inválidas no arquivo original,0
4,Horários inválidos em 2024,0
5,Códigos de infração inválidos em 2024,0
6,Registros sem local,0
7,Duplicidades exatas em todas as colunas,8232
8,Linhas envolvidas em possíveis repetições,15241


Relatório de qualidade corrigido!


In [ ]:
pasta = DRIVE_BASE_PATH

# Tabela principal
fato_infracoes.to_csv(
    f"{pasta}/fato_infracoes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Indicadores
indicador_tipo.to_csv(
    f"{pasta}/indicadores_tipo.csv",
    index=False,
    encoding="utf-8-sig"
)

indicador_horario.to_csv(
    f"{pasta}/indicadores_horario.csv",
    index=False,
    encoding="utf-8-sig"
)

indicador_mes.to_csv(
    f"{pasta}/indicadores_mes.csv",
    index=False,
    encoding="utf-8-sig"
)

indicador_dia_hora.to_csv(
    f"{pasta}/indicadores_dia_hora.csv",
    encoding="utf-8-sig"
)

# Auditoria
auditoria_nulos.to_csv(
    f"{pasta}/auditoria_nulos.csv",
    index=False,
    encoding="utf-8-sig"
)

relatorio_qualidade.to_csv(
    f"{pasta}/relatorio_qualidade.csv",
    index=False,
    encoding="utf-8-sig"
)

# Registro da origem
metadados = {
    "fonte": "Portal de Dados Abertos do Recife",
    "conjunto": "Registro das Infrações de Trânsito",
    "ano": 2024,
    "resource_id": resource_id,
    "data_extracao": datetime.now().isoformat(),
    "formato": "CSV"
}

with open(
    f"{pasta}/metadados.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        metadados,
        arquivo,
        ensure_ascii=False,
        indent=4
    )

print("Arquivos salvos com sucesso!")

Arquivos salvos com sucesso!


In [ ]:
print("ARQUIVOS DO ETL 2:")

for arquivo in os.listdir(pasta):
    print(arquivo)

ARQUIVOS DO ETL 2:
raw
processed
reports
fato_infracoes.csv
indicadores_tipo.csv
indicadores_horario.csv
indicadores_mes.csv
indicadores_dia_hora.csv
auditoria_nulos.csv
relatorio_qualidade.csv
metadados.json


In [ ]:
import shutil

arquivo_zip = shutil.make_archive(
    "ETL2_infracoes",
    "zip",
    pasta
)

from google.colab import files

files.download(arquivo_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("VERIFICAÇÃO DOS LOCAIS")

print(
    df_2024["local_cometimento"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nVERIFICAÇÃO DAS POSSÍVEIS REPETIÇÕES")

print("Linhas envolvidas: ", repetidos.sum())

print("\nTOTAL DE REGISTROS: ", len(df_2024))

VERIFICAÇÃO DOS LOCAIS
local_cometimento
AV. CAXANGÁ, LADO OPOSTO AO Nº 5693, SENTIDO SUBÚRBIO                     10790
AV. GOV. AGAMENON MAGALHÃES, PRÓXIMO AO SEMÁFORO 069, SENT. BOA VIAGEM     9425
AV. CDE. DA BOA VISTA, SEMAFORO 486, Sentido:CENTRO Faixa: 4               8240
AV. CDE. DA BOA VISTA, SEMAFORO 486, Sentido:SUBURBIO Faixa: 2             7982
AV. CDE. DA BOA VISTA, SEMAFORO 486, Sentido:SUBURBIO Faixa: 1             5737
AV. CAXANGA, SEMAFORO 609, Sentido:SUBURBIO Faixa: 3                       5134
AV. CDE. DA BOA VISTA, SEMAFORO 486, Sentido:CENTRO Faixa: 3               4637
AV. CAXANGÁ, EM FRENTE AO Nº 4500, SENTIDO SUBÚRBIO                        4506
RUA ARQUITETO LUIZ NUNES ENTRE O N.315 E 379                               4125
CAIS SANTA RITA, PROX. AO N. 675                                           3978
RUA REAL DA TORRE, SEMAFORO 237, Sentido:AFOGADOSFaixa: 2                  3734
AV. CAXANGÁ, APÓS O Nº 125, SENTIDO CENTRO                                 3707

In [ ]:
# Verificar locais que podem representar ausência de informação

valores_suspeitos = [
    "NÃO INFORMADO",
    "NAO INFORMADO",
    "N/I",
    "NI",
    "N/A",
    "NA",
    "-",
    "SEM LOCAL",
    "NAN"
]

locais = df_2024["local_cometimento"].astype("string").str.strip()

suspeitos = locais.str.upper().isin(valores_suspeitos)

print("LOCAIS SUSPEITOS: ", suspeitos.sum())

print("\nVALORES ENCONTRADOS: ")
print(locais[suspeitos].value_counts())

print("\nDUPLICIDADES EXATAS: ")
print(df_2024.duplicated().sum())

print("\nDUPLICIDADES NOS CAMPOS INVESTIGADOS: ")
print(repetidos.sum())

LOCAIS SUSPEITOS:  0

VALORES ENCONTRADOS: 
Series([], Name: count, dtype: Int64)

DUPLICIDADES EXATAS: 
8232

DUPLICIDADES NOS CAMPOS INVESTIGADOS: 
15241


In [ ]:
# Atualizar relatório de qualidade

novos_indicadores = pd.DataFrame({
    "indicador": [
        "Duplicidades exatas em todas as colunas",
        "Linhas envolvidas em possíveis repetições"
    ],
    "quantidade": [
        df_2024.duplicated().sum(),
        repetidos.sum()
    ]
})

relatorio_qualidade = pd.concat(
    [relatorio_qualidade, novos_indicadores],
    ignore_index=True
)

# Salvar relatório atualizado

relatorio_qualidade.to_csv(
    f"{pasta}/relatorio_qualidade.csv",
    index=False,
    encoding="utf-8-sig"
)

display(relatorio_qualidade)

print("Relatório de qualidade atualizado! ")

,indicador,quantidade
0,Linhas extraídas,404975
1,Linhas carregadas,404975
2,Linhas fora do recorte ou com data inválida,0
3,Datas inválidas no arquivo original,0
4,Horários inválidos em 2024,0
5,Códigos de infração inválidos em 2024,0
6,Registros sem local,0
7,Duplicidades exatas em todas as colunas,8232
8,Linhas envolvidas em possíveis repetições,15241
9,Duplicidades exatas em todas as colunas,8232


Relatório de qualidade atualizado! 
